# Title


**IMDB Movie Review Sentiment Analysis**

# Problem Statement



Nowadays, people share thousands of movie reviews online. These reviews contain different opinions, such as positive and negative feedback.

Manually reading and identifying whether a review is positive or negative takes a lot of time.

Therefore, the problem is to develop a Machine Learning model that automatically analyzes movie reviews and classifies them as Positive or Negative based on the text.

# Main Objective

1. Collect movie reviews from the IMDB dataset.

2. Clean and preprocess the review text.

3. Convert text into numerical features using Word2Vector.

4. Train different ANN Architectures.

5. Compare their performance using F1_Score.

6. Select the best-performing model.

7. Predict whether a new movie review is Positive or Negative.

# Dataset Loading and Data Overivew

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install optuna gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 139.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [optuna]


In [3]:
import pandas as pd
import numpy as np
import re
import html
import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
                                        Dense,
                                        Dropout,
                                        BatchNormalization
                                    )

from sklearn.metrics import accuracy_score, classification_report

# Download required resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [4]:
df = pd.read_csv('/content/drive/MyDrive/ML/imdb sentiment analysis/IMDB Dataset.csv')

In [5]:
# Display first 5 rows
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:


# Check number of rows and columns
print("Shape:", df.shape)

Shape: (50000, 2)


In [7]:
# Check column names
print("Columns:", df.columns.tolist())

Columns: ['review', 'sentiment']


In [8]:
# Check data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


#**Data cleaning**

In [9]:
df.isnull().sum()

,0
review,0
sentiment,0


**Observation**

There is no missing content


**Check duplicate rows**

In [10]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 418


In [11]:


df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

**Observation**

There are totla 418 duplicate values are available so then i remove that values

In [12]:
print(df['sentiment'].value_counts())
print(df['sentiment'].unique())

sentiment
positive    24884
negative    24698
Name: count, dtype: int64
['positive' 'negative']


In [13]:
stop_words = set(stopwords.words('english'))

# Preserve important negation words
stop_words -= {'no', 'nor', 'not'}

lemmatizer = WordNetLemmatizer()


def preprocess_text(text):

    # 1. Handle missing values
    if pd.isna(text):
        return ""

    text = str(text)

    # 2. Decode HTML entities
    text = html.unescape(text)

    # 3. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # 4. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 5. Convert to lowercase
    text = text.lower()

    # 6. Expand common contractions
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bcan't\b", "cannot", text)
    text = re.sub(r"\bshan't\b", "shall not", text)
    text = re.sub(r"\bain't\b", "is not", text)
    text = re.sub(r"n't\b", " not", text)
    text = re.sub(r"'re\b", " are", text)
    text = re.sub(r"'ve\b", " have", text)
    text = re.sub(r"'ll\b", " will", text)
    text = re.sub(r"'m\b", " am", text)
    text = re.sub(r"'d\b", " would", text)

    # 7. Remove punctuation, digits and special characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 8. Remove extra spaces and tokenize
    tokens = text.split()

    # 9. Remove stopwords while preserving negations
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # 10. Lemmatization
    tokens = [
                lemmatizer.lemmatize(word, pos='v')
                for word in tokens
             ]

    # 11. Return processed text
    return tokens

In [14]:
df['processed_review'] = df['review'].apply(preprocess_text)

In [15]:
df[['review', 'processed_review', 'sentiment']].head()

,review,processed_review,sentiment
0,One of the other reviewers has mentioned that ...,"[one, reviewers, mention, watch, oz, episode, ...",positive
1,A wonderful little production. <br /><br />The...,"[wonderful, little, production, film, techniqu...",positive
2,I thought this was a wonderful way to spend ti...,"[think, wonderful, way, spend, time, hot, summ...",positive
3,Basically there's a family where a little boy ...,"[basically, family, little, boy, jake, think, ...",negative
4,"Petter Mattei's ""Love in the Time of Money"" is...","[petter, mattei, love, time, money, visually, ...",positive


# Word2Vector

In [16]:
# Train Word2Vec Method
word2vec_model = Word2Vec(
                           sentences= df['processed_review'],
                           vector_size= 100,
                           window = 5,
                           min_count= 2,
                           sg = 1,
                           epochs= 10
                        )


In [17]:
# Function to convert a review into an average word vector
def review_to_vector(review, model,vector_size):
  vectors = [model.wv[word] for word in review if word in model.wv]
  if vectors:
    return np.mean(vectors, axis=0)
  else:
    return np.zeros(vector_size)

vector_size = 100
review_vectors = np.array([review_to_vector(review, word2vec_model, vector_size) for review in df['processed_review']])
print(review_vectors)
print(review_vectors.shape)


[[-0.18097854  0.08441763 -0.0438321  ... -0.1404001   0.13911808
  -0.07118472]
 [-0.10169485  0.12697133  0.00600713 ... -0.11971977 -0.01446365
  -0.06251806]
 [-0.08499594  0.11966503  0.00258926 ... -0.06817801  0.02382539
  -0.08227953]
 ...
 [-0.11929711  0.11691845  0.00665663 ... -0.11222304  0.15026392
   0.03210952]
 [-0.13866884  0.1699579   0.02256678 ... -0.07471104  0.05508551
  -0.02725601]
 [-0.18039012  0.06373602 -0.1022758  ... -0.05496573  0.01034458
  -0.01850065]]
(49582, 100)


#**Separate features and target**

In [18]:
X = review_vectors
y = df['sentiment']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (49582, 100)
Target shape: (49582,)


#**Encode sentiment labels**

In [19]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

print(le.classes_)

['negative' 'positive']


**Observation:**

Convert positive and negative labels into numerical values.

#**Split the dataset**

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
                                                    X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 39665
Testing samples: 9917


In [21]:
X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)
y_train = pd.DataFrame(y_train)
y_test = pd.DataFrame(y_test)

**Observation:**

We'll use 80% for training ,20% for testing, preserving the sentiment proportions with stratify = y

# ANN Architectures

# Defining metrics


In [22]:
import keras
f1_metric = keras.metrics.F1Score(threshold=0.5, name="f1_score")

# **Architecture 1**

- **Base Model**
- Input layer
- One hidden layer (16 neurons)
- Output layer (1 neuron)

In [23]:
# Shallow Network
model_1 = Sequential([
                      Dense(16, activation='relu', input_shape = (X_train.shape[1],)),
                      Dense(1, activation='sigmoid')
                     ])
# Compile
model_1.compile(
                  optimizer = 'adam',
                  loss = 'binary_crossentropy',
                  metrics = [keras.metrics.F1Score(name='f1_score')]

)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Model Summary**

In [24]:
model_1.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │         1,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,633 (6.38 KB)

 Trainable params: 1,633 (6.38 KB)

 Non-trainable params: 0 (0.00 B)

**Train the Model**

In [25]:
history = model_1.fit(
                       X_train, y_train,
                       validation_split = 0.2,
                       epochs = 20,
                       batch_size = 32
                  )

Epoch 1/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - f1_score: 0.6685 - loss: 0.4690 - val_f1_score: 0.6678 - val_loss: 0.3575
Epoch 2/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - f1_score: 0.6685 - loss: 0.3267 - val_f1_score: 0.6678 - val_loss: 0.3163
Epoch 3/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - f1_score: 0.6685 - loss: 0.3076 - val_f1_score: 0.6679 - val_loss: 0.3071
Epoch 4/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - f1_score: 0.6685 - loss: 0.3010 - val_f1_score: 0.6680 - val_loss: 0.3033
Epoch 5/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - f1_score: 0.6685 - loss: 0.2983 - val_f1_score: 0.6680 - val_loss: 0.3010
Epoch 6/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - f1_score: 0.6685 - loss: 0.2961 - val_f1_score: 0.6680 - val_loss: 0.3002
Epoch 7/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - f1_score: 0.6685 - loss: 0.2946 - val_f1_score: 0.6680 - val_loss: 0.2992
Epoch 8/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - f1_score: 0.6685 - loss: 0.2934 - val_f1_score: 0.

In [26]:
# Evaluate the Model
loss, arc1_F1_Score = model_1.evaluate(X_test, y_test)
print('Test Loss :',loss)
print('Test F1_Score : ', arc1_F1_Score)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.6683 - loss: 0.3016
Test Loss : 0.3016407787799835
Test F1_Score :  [0.6683228]


# **Acrhitecture 2**
- Input layer
- Hidden layer 1: 32 neurons
- Hidden layer 2: 16 neurons
- Output layer

In [27]:
model_2 = Sequential([
                       Dense(32, activation='relu', input_shape=(X_train.shape[1], )),
                       Dense(16, activation='relu'),
                       Dense(1, activation='sigmoid')
                      ])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
# Compile the model
model_2.compile(
                 optimizer = 'adam',
                 loss = 'binary_crossentropy',
                 metrics = [f1_metric]
)

In [29]:
# Model summary
model_2.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 32)             │         3,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,777 (14.75 KB)

 Trainable params: 3,777 (14.75 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
# Train the model
history = model_2.fit(X_train, y_train,
                      validation_data = (X_test, y_test),
                      epochs = 20,
                      batch_size = 32
                    )

Epoch 1/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - f1_score: 0.8446 - loss: 0.3679 - val_f1_score: 0.8601 - val_loss: 0.3146
Epoch 2/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - f1_score: 0.8727 - loss: 0.3023 - val_f1_score: 0.8778 - val_loss: 0.3029
Epoch 3/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8758 - loss: 0.2977 - val_f1_score: 0.8798 - val_loss: 0.2982
Epoch 4/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8763 - loss: 0.2948 - val_f1_score: 0.8789 - val_loss: 0.2963
Epoch 5/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - f1_score: 0.8789 - loss: 0.2928 - val_f1_score: 0.8828 - val_loss: 0.2970
Epoch 6/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - f1_score: 0.8792 - loss: 0.2908 - val_f1_score: 0.8644 - val_loss: 0.3049
Epoch 7/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - f1_score: 0.8789 - loss: 0.2902 - val_f1_score: 0.8839 - val_loss: 0.2933
Epoch 8/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8805 - loss: 0.288

In [31]:
# Evaluate the model
loss, arch2_F1_Score = model_2.evaluate(X_test, y_test)
print('Test Loss : ',loss)
print('Test F1_Score',arch2_F1_Score)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - f1_score: 0.8764 - loss: 0.2853
Test Loss :  0.28529495000839233
Test F1_Score [0.87639767]


# **Architecture 3 :**

- Input layer
- Hidden layer 1: 128 neurons
- Hidden layer 2: 64 neurons
- Hidden layer 3: 32 neurons
- Hidden layer 4: 16 neurons
- Output layer

In [32]:
# Deep Network
model_3 = Sequential([
                    Dense(128,activation='relu', input_shape= (X_train.shape[1],)),
                    Dense(64, activation = 'relu'),
                    Dense(32, activation = 'relu'),
                    Dense(16, activation = 'relu'),
                    Dense(1, activation='sigmoid')
                  ])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [33]:
# Compile
model_3.compile(
                 optimizer = 'adam',
                 loss = 'binary_crossentropy',
                 metrics = [f1_metric]
)

In [34]:
# Model summary
model_3.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_5 (Dense)                 │ (None, 128)            │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,809 (93.00 KB)

 Trainable params: 23,809 (93.00 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
# Train the model
history = model_3.fit(
                        X_train,y_train,
                        validation_data = (X_test, y_test),
                        epochs = 20,
                        batch_size = 32
                      )

Epoch 1/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - f1_score: 0.8559 - loss: 0.3444 - val_f1_score: 0.8777 - val_loss: 0.3090
Epoch 2/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - f1_score: 0.8727 - loss: 0.3020 - val_f1_score: 0.8792 - val_loss: 0.3099
Epoch 3/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - f1_score: 0.8749 - loss: 0.2986 - val_f1_score: 0.8687 - val_loss: 0.3507
Epoch 4/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8770 - loss: 0.2950 - val_f1_score: 0.8715 - val_loss: 0.2944
Epoch 5/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8808 - loss: 0.2901 - val_f1_score: 0.8745 - val_loss: 0.3261
Epoch 6/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8797 - loss: 0.2872 - val_f1_score: 0.8730 - val_loss: 0.2942
Epoch 7/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - f1_score: 0.8794 - loss: 0.2861 - val_f1_score: 0.8732 - val_loss: 0.2940
Epoch 8/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8826 - loss: 0.2836 -

In [36]:
# Evaluate the model
loss, arch3_F1_Score = model_3.evaluate(X_test, y_test)
print('Test Loss : ', loss)
print('Test F1 Score :',arch3_F1_Score)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - f1_score: 0.8746 - loss: 0.2903
Test Loss :  0.2903133034706116
Test F1 Score : [0.8745928]


# **Architecture 4 :**

- Input layer
- Hidden layer 1: 256 neurons
- Hidden layer 2: 128 neurons
- Hidden layer 3: 64 neurons
- Hidden layer 4: 1 neurons
- Output layer

In [37]:
# Wide Network
model_4 = Sequential([
                       Dense(256, activation = 'relu', input_shape = (X_train.shape[1],)),
                       Dense(128, activation= 'relu'),
                       Dense(64, activation = 'relu'),
                       Dense(1, activation = 'sigmoid')
                    ])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [38]:
# compile
model_4.compile(
                optimizer ='adam',
                loss = 'binary_crossentropy',
                metrics = [f1_metric])

In [39]:
# Model summary
model_4.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 256)            │        25,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,073 (262.00 KB)

 Trainable params: 67,073 (262.00 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
# Train the model
history = model_4.fit(
                      X_train, y_train,
                      validation_data = (X_test, y_test),
                      epochs = 20,
                      batch_size = 32
                     )

Epoch 1/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - f1_score: 0.8597 - loss: 0.3319 - val_f1_score: 0.8794 - val_loss: 0.2979
Epoch 2/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8730 - loss: 0.3010 - val_f1_score: 0.8753 - val_loss: 0.2952
Epoch 3/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8753 - loss: 0.2986 - val_f1_score: 0.8600 - val_loss: 0.3612
Epoch 4/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8783 - loss: 0.2933 - val_f1_score: 0.8824 - val_loss: 0.2934
Epoch 5/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8779 - loss: 0.2888 - val_f1_score: 0.8729 - val_loss: 0.2924
Epoch 6/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8795 - loss: 0.2877 - val_f1_score: 0.8843 - val_loss: 0.2939
Epoch 7/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - f1_score: 0.8831 - loss: 0.2832 - val_f1_score: 0.8859 - val_loss: 0.2922
Epoch 8/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - f1_score: 0.8841 - loss: 0.2806 - 

In [41]:
# Evaluate the model
loss, arc4_F1_Score = model_4.evaluate(
                                        X_test, y_test
                                      )
print('Test Loss : ', loss)
print('Test F1 Score :',arc4_F1_Score)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - f1_score: 0.8861 - loss: 0.2934
Test Loss :  0.2933659851551056
Test F1 Score : [0.88609046]


# **Architecture 5 :**

- Input layer
- Hidden layer 1: 128 neurons
- BatchNormalization
- Dropout
- Hidden layer 2: 64 neurons
- Hidden layer 3: 32 neurons
- Hidden layer 4: 1 neurons
- Output layer

In [42]:
model_5 = Sequential([
                       Dense(128, activation= 'relu', input_shape = (X_train.shape[1],)),
                       BatchNormalization(),
                       Dropout(0.3),
                       Dense(64, activation = 'relu'),
                       Dense(32, activation ='relu'),
                       Dense(1, activation= 'sigmoid')
                    ])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [43]:
# compile
model_5.compile(
                optimizer ='adam',
                loss = 'binary_crossentropy',
                metrics = [f1_metric]
               )

In [44]:
# model summary
model_5.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                │ (None, 128)            │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,809 (93.00 KB)

 Trainable params: 23,553 (92.00 KB)

 Non-trainable params: 256 (1.00 KB)

In [45]:
# Train the model
history = model_5.fit(
                       X_train, y_train,
                       validation_data = (X_test, y_test),
                       epochs = 20,
                       batch_size = 32
                      )

Epoch 1/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - f1_score: 0.8608 - loss: 0.3409 - val_f1_score: 0.8753 - val_loss: 0.3029
Epoch 2/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8680 - loss: 0.3142 - val_f1_score: 0.8684 - val_loss: 0.3097
Epoch 3/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8683 - loss: 0.3112 - val_f1_score: 0.8774 - val_loss: 0.3103
Epoch 4/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8732 - loss: 0.3064 - val_f1_score: 0.8777 - val_loss: 0.2959
Epoch 5/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8703 - loss: 0.3089 - val_f1_score: 0.8813 - val_loss: 0.2891
Epoch 6/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8734 - loss: 0.3045 - val_f1_score: 0.8814 - val_loss: 0.3007
Epoch 7/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8716 - loss: 0.3051 - val_f1_score: 0.8840 - val_loss: 0.2966
Epoch 8/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8738 - loss: 0.3033 -

In [46]:
# Evaluate the model
loss, arch5_F1_Score = model_5.evaluate(X_test, y_test)
print('Test Loss : ', loss)
print('Test F1 Score :',arch5_F1_Score)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - f1_score: 0.8825 - loss: 0.2897
Test Loss :  0.2896919846534729
Test F1 Score : [0.88251364]


# Architecture 6

- Hyperparameter Tuning

In [47]:
def hyperparameter(trail):
  hidden_units = trail.suggest_int('hidden_units', 64, 512)
  dropout_rate = trail.suggest_float('dropout_rate', 0.1, 0.5)
  learning_rate = trail.suggest_float('learning_rate', 1e-4, 1e-2, log = True)
  optimizer_name = trail.suggest_categorical('optimizer', ['adam', 'sgd'])

  # Model Defnition
  model_6 = keras.Sequential([
                             layers.Input(shape = (X_train.shape[1],)),
                             layers.Dense(hidden_units, activation = 'relu'),
                             layers.Dropout(dropout_rate),
                             layers.Dense(1, activation = 'sigmoid')
                          ])
  if optimizer_name == 'sgd':
    optimizer = getattr(keras.optimizers, optimizer_name.upper())(learning_rate = learning_rate)
  else:
    optimizer = getattr(keras.optimizers, optimizer_name.title())(learning_rate = learning_rate)

  # Model compile
  model_6.compile(
                 optimizer = optimizer,
                 loss = 'binary_crossentropy',
                 metrics = [f1_metric]
               )

  # Train the model
  history = model_6.fit(
                         X_train, y_train,
                         validation_split = 0.2,
                         epochs = 100,
                         batch_size = 128,
                         verbose= 0
                      )

  # Returning the best validation f1 score
  return float(max(float(x[0]) for x in history.history['val_f1_score']))

In [48]:
pip install optuna

In [49]:
import optuna

In [50]:
study = optuna.create_study(direction = 'maximize')
study.optimize(hyperparameter, n_trials = 20)
print('Best Trial:')
trial = study.best_trial
arch6_F1_Score = trial.value
print(f'F1 Score : {arch6_F1_Score}')
for key, value in trial.params.items():
  print(f'{key} : {value}')

[I 2026-11-05 09:14:31,472] A new study created in memory with name: no-name-7590013c-0da6-432f-bf87-7348e75a19d4
[I 2026-11-05 09:16:22,231] Trial 0 finished with value: 0.883385181427002 and parameters: {'hidden_units': 239, 'dropout_rate': 0.1823655587710401, 'learning_rate': 0.0002126597043038064, 'optimizer': 'adam'}. Best is trial 0 with value: 0.883385181427002.
[I 2026-11-05 09:18:00,491] Trial 1 finished with value: 0.7806410193443298 and parameters: {'hidden_units': 470, 'dropout_rate': 0.4959466694029159, 'learning_rate': 0.0003800684278020189, 'optimizer': 'sgd'}. Best is trial 0 with value: 0.883385181427002.
[I 2026-11-05 09:19:31,429] Trial 2 finished with value: 0.7915672659873962 and parameters: {'hidden_units': 231, 'dropout_rate': 0.18765700326983836, 'learning_rate': 0.0007942403761672786, 'optimizer': 'sgd'}. Best is trial 0 with value: 0.883385181427002.
[I 2026-11-05 09:21:02,710] Trial 3 finished with value: 0.8493422269821167 and parameters: {'hidden_units': 38

Best Trial:
F1 Score : 0.8889435529708862
hidden_units : 447
dropout_rate : 0.10168301057795652
learning_rate : 0.001166826389287164
optimizer : adam


In [51]:
best_params = study.best_trial.params
hidden_units = best_params['hidden_units']
dropout_rate = best_params['dropout_rate']
learning_rate = best_params['learning_rate']
optimizer_name = best_params['optimizer']
print('Best Hyperparameters :', best_params)

Best Hyperparameters : {'hidden_units': 447, 'dropout_rate': 0.10168301057795652, 'learning_rate': 0.001166826389287164, 'optimizer': 'adam'}


In [52]:
# Model building
model_6 = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(hidden_units, activation='relu'),
    layers.Dropout(dropout_rate),

    layers.Dense(1, activation='sigmoid')
])

# Select optimizer
if optimizer_name == 'adam':
    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )
else:
    optimizer = keras.optimizers.SGD(
        learning_rate=learning_rate
    )

# Compile the model
model_6.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=[f1_metric]
)

# Display summary
model_6.summary()

Model: "sequential_25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_58 (Dense)                │ (None, 447)            │        45,147 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 447)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_59 (Dense)                │ (None, 1)              │           448 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,595 (178.11 KB)

 Trainable params: 45,595 (178.11 KB)

 Non-trainable params: 0 (0.00 B)

In [53]:
# Train the model
history = model_6.fit(
                    X_train,
                    y_train,
                    validation_split = 0.2,
                    batch_size=128,
                    epochs=100,
                    verbose=1
                  )

Epoch 1/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - f1_score: 0.8402 - loss: 0.4084 - val_f1_score: 0.8674 - val_loss: 0.3191
Epoch 2/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8715 - loss: 0.3065 - val_f1_score: 0.8693 - val_loss: 0.3045
Epoch 3/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8745 - loss: 0.2989 - val_f1_score: 0.8755 - val_loss: 0.2990
Epoch 4/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8757 - loss: 0.2953 - val_f1_score: 0.8766 - val_loss: 0.3067
Epoch 5/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8768 - loss: 0.2935 - val_f1_score: 0.8769 - val_loss: 0.2989
Epoch 6/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8769 - loss: 0.2920 - val_f1_score: 0.8765 - val_loss: 0.2956
Epoch 7/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8779 - loss: 0.2914 - val_f1_score: 0.8776 - val_loss: 0.2972
Epoch 8/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8807 - loss: 0.2881 - val_f1_

In [54]:
# Evaluate
loss, arch6_F1_Score = model_6.evaluate(
                                          X_test,
                                          y_test,
                                         )

print(f'Test Loss: {loss}')
print(f'Test Accuracy: {arch6_F1_Score}')

310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - f1_score: 0.8773 - loss: 0.2941
Test Loss: 0.2940881848335266
Test Accuracy: [0.8772962]


# DataFrame


In [55]:
# Creating all architecture F1 Scores DataFrame
df = pd.DataFrame({
                  'Architecture': [
                              'Architecture 1 (Shallow Network)',
                              'Architecture 2 ',
                              'Architecture 3 (Deep Network)',
                              'Architecture 4 (Wide Network) ',
                              'Architecture 5 (BatchNormalization & Dropout)',
                              'Architecture 6 (Hyperparameter Tuning)'
                              ],

                  'F1_Scores': [
                              arc1_F1_Score,
                              arch2_F1_Score,
                              arch3_F1_Score,
                              arc4_F1_Score,
                              arch5_F1_Score,
                              arch6_F1_Score
                              ]
                  })



In [56]:
df = df.sort_values(by='F1_Scores', ascending= False).reset_index(drop=True)
df['F1_Scores'] = df['F1_Scores']*100
df

,Architecture,F1_Scores
0,Architecture 4 (Wide Network),[88.60905]
1,Architecture 5 (BatchNormalization & Dropout),[88.251366]
2,Architecture 6 (Hyperparameter Tuning),[87.72962]
3,Architecture 2,[87.63977]
4,Architecture 3 (Deep Network),[87.459274]
5,Architecture 1 (Shallow Network),[66.83228]


**Observation:**

Out all 6 architectures iam choosing the 2nd architecture cazz out of all architectures in this architecture parameter is simple and it gives better perfomance.

# Saving the Model

**Save your best model**

In [58]:
import joblib

model_2.save("IMDB_Sentiment_Analysis_(Word2Vec&ANN).keras")
word2vec_model.save('imdb_word2vec.model')
joblib.dump(le, 'imdb_label_encoder.pkl')

['imdb_label_encoder.pkl']